# NFN — Neural Fractal Network · Kaggle Training

**Free GPU:** T4 (16 GB) or P100 (16 GB) · **Quota:** 30 h/week

### Instructions
1. Enable GPU: `Settings → Accelerator → GPU T4 x1`
2. Run all cells top to bottom (`Run All`)
3. Checkpoints saved to `/kaggle/working/checkpoints/` (auto-downloaded when session ends)
4. Training survives up to **12h** per session — resume with the last cell

> **Preset auto-selected:** `medium_wikipedia` (Simple Wikipedia, ~85M param model, ~12h on T4)

In [ ]:
# ── Cell 1: Clone repo & install deps ────────────────────────────────────────
import os

if not os.path.exists('/kaggle/working/FNN'):
    !git clone https://github.com/AFKmoney/FNN.git /kaggle/working/FNN
else:
    !git -C /kaggle/working/FNN pull

os.chdir('/kaggle/working/FNN')
!pip install -e . -q
!pip install tqdm -q
print('✓ Repo ready')

In [ ]:
# ── Cell 2: Check GPU ─────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {gpu}  ({vram:.1f} GB VRAM)')
else:
    print('⚠ No GPU detected — go to Settings → Accelerator → GPU T4 x1')

print(f'PyTorch {torch.__version__}')

In [ ]:
# ── Cell 3: Download dataset ──────────────────────────────────────────────────
import sys
sys.path.insert(0, '/kaggle/working/FNN')

from datasets.downloader import DatasetDownloader

dl = DatasetDownloader('/kaggle/working/FNN/data')
print('Downloading Simple Wikipedia (~120 MB)...')
text = dl.get('wikipedia-en-simple', max_chars=50_000_000)  # 50M chars
print(f'✓ Dataset ready: {len(text):,} characters')

# Save to file for training
os.makedirs('/kaggle/working/FNN/data', exist_ok=True)
with open('/kaggle/working/FNN/data/train.txt', 'w') as f:
    f.write(text)
print('✓ Saved to data/train.txt')

In [ ]:
# ── Cell 4: Train ─────────────────────────────────────────────────────────────
# Runs in this cell — Kaggle sessions last up to 12h
# Checkpoints saved every 500 steps automatically

import os
os.chdir('/kaggle/working/FNN')
os.makedirs('checkpoints', exist_ok=True)

# Check if resuming
resume_flag = ''
if os.path.exists('checkpoints/agi_nfn_latest.pt'):
    resume_flag = '--resume checkpoints/agi_nfn_latest.pt'
    print('▶ Resuming from checkpoint...')
else:
    print('▶ Starting fresh training...')

# medium config — fits in 16GB with batch=6, seq=512
!python train_agi.py \
    --text data/train.txt \
    --config medium \
    --epochs 3 \
    --batch 6 \
    --seq-len 512 \
    --lr 2e-4 \
    --fp16 \
    --sample-every 500 \
    --save-every 500 \
    {resume_flag}

In [ ]:
# ── Cell 5: Quick generation test ─────────────────────────────────────────────
import torch, sys
sys.path.insert(0, '/kaggle/working/FNN')

from nfn.agi_model import build_agi_model
from nfn.tokenizer import NFNTokenizer

tok = NFNTokenizer()
ckpt = torch.load('checkpoints/agi_nfn_final.pt', map_location='cpu')
model = build_agi_model(vocab_size=tok.vocab_size, d_model=512, n_blocks=8)
model.load_state_dict(ckpt['model_state'])
model.eval()

prompt = "The universe is"
ids = tok.encode(prompt, add_bos=True)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=100, temperature=0.8)
print(tok.decode(out[0].tolist()))

In [ ]:
# ── Cell 6: Resume in next session ───────────────────────────────────────────
# If your session ended, run cells 1-2 then this cell to continue
import os
os.chdir('/kaggle/working/FNN')

if os.path.exists('checkpoints/agi_nfn_latest.pt'):
    !python train_agi.py \
        --text data/train.txt \
        --config medium \
        --batch 6 \
        --seq-len 512 \
        --fp16 \
        --epochs 3 \
        --resume checkpoints/agi_nfn_latest.pt
else:
    print('No checkpoint found — run cells 3-4 first')